In [ ]:
%pip install torch datasets numpy matplotlib

In [2]:

# Import required libraries

import torch
import torch.nn as nn
from torch.nn import functional as F

import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

import random
import os
from pathlib import Path

print("All libraries imported successfully!")


All libraries imported successfully!


In [3]:

# Set random seeds for reproducibility

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Select available computing device

if torch.backends.mps.is_available():
    device = 'mps'

elif torch.cuda.is_available():
    device = 'cuda'

else:
    device = 'cpu'

print("Using device:", device)


Using device: cuda


In [4]:

# Load training dataset

train_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train[:20000]"
)

# Load validation dataset

validation_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="validation[:2000]"
)

# Extract stories

train_stories = [
    story["text"]
    for story in train_dataset
    if story["text"].strip()
]

validation_stories = [
    story["text"]
    for story in validation_dataset
    if story["text"].strip()
]

# Display dataset information

print("Training stories:", len(train_stories))
print("Validation stories:", len(validation_stories))

print("\nSample Story:\n")
print(train_stories[0])


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Training stories: 20000
Validation stories: 2000

Sample Story:

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [5]:
# Create vocabulary from training and validation data

train_text = "\n\n".join(train_stories)
validation_text = "\n\n".join(validation_stories)

chars = sorted(list(set(train_text + validation_text)))

# Total number of unique characters
vocab_size = len(chars)

# Character to integer mapping
stoi = {ch: i for i, ch in enumerate(chars)}

# Integer to character mapping
itos = {i: ch for i, ch in enumerate(chars)}

# Encoder: Convert text into numbers
def encode(text):
    return [stoi[ch] for ch in text]

# Decoder: Convert numbers back into text
def decode(tokens):
    return ''.join([itos[i] for i in tokens])


# Display vocabulary information

print("Vocabulary size:", vocab_size)

print("\nVocabulary characters:")
print(chars)

# Test tokenizer

sample_text = "Hello world!"

encoded_text = encode(sample_text)
decoded_text = decode(encoded_text)

print("\nOriginal text:", sample_text)
print("Encoded text:", encoded_text)
print("Decoded text:", decoded_text)

Vocabulary size: 98

Vocabulary characters:
['\n', ' ', '!', '"', '$', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\xa0', '¡', '¦', '©', '«', '±', '³', '»', 'Â', 'Ã', 'â', 'œ', 'Š', '˜', '“', '”', '€', '™']

Original text: Hello world!
Encoded text: [35, 58, 65, 65, 68, 1, 76, 68, 71, 65, 57, 2]
Decoded text: Hello world!


In [6]:

# Encode training and validation text

train_ids = encode(train_text)
validation_ids = encode(validation_text)

# Convert encoded data into PyTorch tensors

train_data = torch.tensor(
    train_ids,
    dtype=torch.long
)

validation_data = torch.tensor(
    validation_ids,
    dtype=torch.long
)

# Display tensor information

print("Training tensor shape:", train_data.shape)
print("Validation tensor shape:", validation_data.shape)

print("\nFirst 20 training tokens:")
print(train_data[:20])


Training tensor shape: torch.Size([17930904])
Validation tensor shape: torch.Size([1595940])

First 20 training tokens:
tensor([42, 67, 58,  1, 57, 54, 78, 11,  1, 54,  1, 65, 62, 73, 73, 65, 58,  1,
        60, 62])


In [7]:

# GPT Model Configuration

batch_size = 32
block_size = 128

max_iters = 3000
eval_interval = 300
learning_rate = 3e-4

eval_iters = 50

n_embd = 128
n_head = 4
n_layer = 4

dropout = 0.1

print("GPT Configuration")
print("------------------------")
print("Vocabulary size:", vocab_size)
print("Batch size:", batch_size)
print("Block size:", block_size)
print("Embedding dimension:", n_embd)
print("Attention heads:", n_head)
print("Transformer layers:", n_layer)
print("Dropout:", dropout)
print("Training iterations:", max_iters)


GPT Configuration
------------------------
Vocabulary size: 98
Batch size: 32
Block size: 128
Embedding dimension: 128
Attention heads: 4
Transformer layers: 4
Dropout: 0.1
Training iterations: 3000


In [8]:

def get_batch(split):

    # Select training or validation data
    data = train_data if split == "train" else validation_data

    # Generate random starting positions
    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    # Create input sequences
    x = torch.stack([
        data[i:i + block_size] for i in ix
    ])

    # Create target sequences (shifted by one character)
    y = torch.stack([
        data[i + 1:i + block_size + 1] for i in ix
    ])

    # Move tensors to selected device
    x = x.to(device)
    y = y.to(device)

    return x, y


# Test the batch function
xb, yb = get_batch("train")

print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)

print("\nExample input:")
print(decode(xb[0].tolist()))

print("\nExample target:")
print(decode(yb[0].tolist()))


Input batch shape: torch.Size([32, 128])
Target batch shape: torch.Size([32, 128])

Example input:
 with their mom and dad. They liked to swim, splash and play in the water. They also liked to see the ducks, fish and frogs that

Example target:
with their mom and dad. They liked to swim, splash and play in the water. They also liked to see the ducks, fish and frogs that 


In [9]:

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        # Key, Query and Value layers
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Causal mask: prevents looking at future characters
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        # Generate keys and queries
        k = self.key(x)
        q = self.query(x)

        # Calculate attention scores
        weights = q @ k.transpose(-2, -1)

        # Scale attention scores
        weights = weights * (k.shape[-1] ** -0.5)

        # Prevent access to future positions
        weights = weights.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        # Convert scores into probabilities
        weights = F.softmax(weights, dim=-1)

        weights = self.dropout(weights)

        # Generate weighted values
        v = self.value(x)
        output = weights @ v

        return output


# Test single attention head
head_size = n_embd // n_head

single_head = Head(head_size).to(device)

sample_input = torch.randn(batch_size, block_size, n_embd).to(device)

output = single_head(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 32])


In [10]:

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        # Create multiple attention heads
        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        # Dropout after concatenation
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Run all attention heads
        outputs = [head(x) for head in self.heads]

        # Concatenate outputs
        out = torch.cat(outputs, dim=-1)

        out = self.dropout(out)

        return out


# Test Multi-Head Attention
multi_head = MultiHeadAttention(
    n_head,
    n_embd // n_head
).to(device)

sample_input = torch.randn(
    batch_size, block_size, n_embd
).to(device)

output = multi_head(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [11]:

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.network = nn.Sequential(

            # Expand dimensions
            nn.Linear(n_embd, 4 * n_embd),

            # Apply ReLU activation
            nn.ReLU(),

            # Reduce dimensions
            nn.Linear(4 * n_embd, n_embd),

            # Apply dropout
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)


# Test Feed Forward Network
feed_forward = FeedForward(n_embd).to(device)

sample_input = torch.randn(
    batch_size, block_size, n_embd
).to(device)

output = feed_forward(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [12]:

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        # Multi-head attention
        self.sa = MultiHeadAttention(n_head, head_size)

        # Feed forward network
        self.ffwd = FeedForward(n_embd)

        # Layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Attention with residual connection
        x = x + self.sa(self.ln1(x))

        # Feed forward with residual connection
        x = x + self.ffwd(self.ln2(x))

        return x


# Test Transformer Block
transformer_block = Block(n_embd, n_head).to(device)

sample_input = torch.randn(
    batch_size, block_size, n_embd
).to(device)

output = transformer_block(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [13]:

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embedding
        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        # Position embedding
        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        # Transformer blocks
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head)
            for _ in range(n_layer)
        ])

        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Output layer
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(
                module.weight, mean=0.0, std=0.02
            )

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(
                module.weight, mean=0.0, std=0.02
            )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Position embeddings
        pos = torch.arange(T, device=device)
        pos_emb = self.position_embedding_table(pos)

        # Combine embeddings
        x = tok_emb + pos_emb

        # Pass through Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Predict next-character logits
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Keep only the latest context
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, loss = self(idx_cond)

            # Consider the last character's prediction
            logits = logits[:, -1, :]

            # Convert logits into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next character
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append predicted character
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# Create GPT model
model = GPTLanguageModel().to(device)

print("GPT model created successfully!")

# Count parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

print("Total parameters:", total_params)


GPT model created successfully!
Total parameters: 767330


In [14]:

# Get a sample batch
xb, yb = get_batch("train")

# Pass input through GPT
logits, loss = model(xb, yb)

print("Input shape:", xb.shape)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())


Input shape: torch.Size([32, 128])
Logits shape: torch.Size([4096, 98])
Initial loss: 4.610427379608154


In [15]:

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

print("Optimizer created successfully!")
print("Learning rate:", learning_rate)


Optimizer created successfully!
Learning rate: 0.0003


In [16]:

@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(
                "train" if split == "train" else "val"
            )

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        results[split] = losses.mean().item()

    model.train()

    return results


print("Initial train and validation loss:")
print(estimate_loss())


Initial train and validation loss:
{'train': 4.614664077758789, 'val': 4.614539623260498}


In [17]:

import time

# Start training timer
start_time = time.time()

model.train()

for iter in range(100):

    # Get a training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Reset gradients
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Update model parameters
    optimizer.step()

    # Print progress
    if (iter + 1) % 20 == 0:
        print(
            f"Iteration {iter + 1}/100 | "
            f"Loss: {loss.item():.4f}"
        )

end_time = time.time()

print("\n100 iterations completed!")
print("Time taken:", round(end_time - start_time, 2), "seconds")


Iteration 20/100 | Loss: 3.5238
Iteration 40/100 | Loss: 3.2254
Iteration 60/100 | Loss: 3.0193
Iteration 80/100 | Loss: 2.8176
Iteration 100/100 | Loss: 2.7028

100 iterations completed!
Time taken: 4.55 seconds


In [18]:

final_losses = estimate_loss()

print("Current model performance")
print("--------------------------")
print("Training loss:", final_losses["train"])
print("Validation loss:", final_losses["val"])


Current model performance
--------------------------
Training loss: 2.666577100753784
Validation loss: 2.6553945541381836


In [19]:

import time

remaining_iters = 2900

start_time = time.time()

model.train()

for iter in range(remaining_iters):

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    current_iter = iter + 101

    # Print progress every 100 iterations
    if current_iter % 100 == 0:
        print(
            f"Iteration {current_iter}/3000 | "
            f"Loss: {loss.item():.4f}"
        )

    # Save checkpoint every 500 iterations
    if current_iter % 500 == 0:

        torch.save({
            "iteration": current_iter,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": loss.item()
        }, "storygpt_checkpoint.pth")

        print("Checkpoint saved at iteration", current_iter)


# Save final checkpoint
torch.save({
    "iteration": 3000,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item()
}, "storygpt_checkpoint.pth")

print("\nTraining completed!")
print("Total time:", round(time.time() - start_time, 2), "seconds")



Iteration 200/3000 | Loss: 2.4195
Iteration 300/3000 | Loss: 2.2887
Iteration 400/3000 | Loss: 2.2278
Iteration 500/3000 | Loss: 2.1551
Checkpoint saved at iteration 500
Iteration 600/3000 | Loss: 2.0680
Iteration 700/3000 | Loss: 1.9542
Iteration 800/3000 | Loss: 1.8711
Iteration 900/3000 | Loss: 1.7274
Iteration 1000/3000 | Loss: 1.6100
Checkpoint saved at iteration 1000
Iteration 1100/3000 | Loss: 1.6322
Iteration 1200/3000 | Loss: 1.5888
Iteration 1300/3000 | Loss: 1.4985
Iteration 1400/3000 | Loss: 1.4975
Iteration 1500/3000 | Loss: 1.4246
Checkpoint saved at iteration 1500
Iteration 1600/3000 | Loss: 1.3871
Iteration 1700/3000 | Loss: 1.4281
Iteration 1800/3000 | Loss: 1.4288
Iteration 1900/3000 | Loss: 1.3878
Iteration 2000/3000 | Loss: 1.3393
Checkpoint saved at iteration 2000
Iteration 2100/3000 | Loss: 1.3398
Iteration 2200/3000 | Loss: 1.2881
Iteration 2300/3000 | Loss: 1.2857
Iteration 2400/3000 | Loss: 1.2769
Iteration 2500/3000 | Loss: 1.2327
Checkpoint saved at iteration

In [20]:

final_losses = estimate_loss()

print("FINAL MODEL EVALUATION")
print("----------------------")
print("Training Loss:", round(final_losses["train"], 4))
print("Validation Loss:", round(final_losses["val"], 4))


FINAL MODEL EVALUATION
----------------------
Training Loss: 1.1314
Validation Loss: 1.1341


In [21]:

final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "vocab_size": vocab_size,
    "chars": chars,
    "stoi": stoi,
    "itos": itos,
    "config": {
        "batch_size": batch_size,
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "training_iterations": 3000
    },
    "final_train_loss": final_losses["train"],
    "final_validation_loss": final_losses["val"]
}

torch.save(
    final_checkpoint,
    "StoryGPT_Final_Checkpoint.pth"
)

print("Final model saved successfully!")

import os
print("Checkpoint size:",
      round(os.path.getsize("StoryGPT_Final_Checkpoint.pth") / (1024 * 1024), 2),
      "MB")


Final model saved successfully!
Checkpoint size: 9.9 MB


In [22]:
# Load the saved checkpoint
checkpoint = torch.load(
    "StoryGPT_Final_Checkpoint.pth",
    map_location=device,
    weights_only=False
)

# Restore model weights
model.load_state_dict(checkpoint["model_state_dict"])

# Restore optimizer state
optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

# Set model to evaluation mode
model.eval()

print("Trained StoryGPT model loaded successfully!")
print("Training iterations:", checkpoint["config"]["training_iterations"])
print("Final training loss:", checkpoint["final_train_loss"])
print("Final validation loss:", checkpoint["final_validation_loss"])

Trained StoryGPT model loaded successfully!
Training iterations: 3000
Final training loss: 1.1313812732696533
Final validation loss: 1.1341285705566406


In [23]:

from google.colab import files

files.download("StoryGPT_Final_Checkpoint.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:

model.eval()

prompt = "Once upon a time"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

generated_tokens = model.generate(
    context,
    max_new_tokens=500
)

generated_story = decode(
    generated_tokens[0].tolist()
)

print("GENERATED STORY")
print("================")
print(generated_story)


GENERATED STORY
Once upon a time that belrach a nugge, stop the whocket animals not happy. So the farm crys and looked.

Lily was in the ball saw Anna. She saw the friendly and loud at the paturn other an inside. He had a who wanted to stree.

"My, thank you!" The liot recross, "Let's importantic!"

Tomy said "It's so like to much bittle girl house and wring urtange to the park. You have together beint mean's own."

"Let's hair. I am we very plase." Tim says, "Thank you having to shore coolde,"

The little girl ran look, Lily'


In [25]:

@torch.no_grad()
def generate_with_temperature(prompt, max_new_tokens, temperature):

    model.eval()

    # Encode the prompt
    context = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Keep the latest context
        idx_cond = context[:, -block_size:]

        # Get model predictions
        logits, loss = model(idx_cond)

        # Get predictions for the last character
        logits = logits[:, -1, :]

        # Apply temperature
        logits = logits / temperature

        # Convert logits into probabilities
        probabilities = F.softmax(logits, dim=-1)

        # Sample the next character
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        # Append new character
        context = torch.cat(
            (context, next_token),
            dim=1
        )

    return decode(context[0].tolist())


# Compare different temperatures
prompt = "Once upon a time"

for temp in [0.5, 0.8, 1.0]:

    print("\n" + "=" * 60)
    print("TEMPERATURE:", temp)
    print("=" * 60)

    story = generate_with_temperature(
        prompt,
        max_new_tokens=400,
        temperature=temp
    )

    print(story)



TEMPERATURE: 0.5
Once upon a time, there was a little girl named Lily. One day, Lily loved to play with the park. She was a big man on the park. She was so excited to her other house and she wanted to play with him.

Once upon a time there was a little girl named Tim saw that he did not go home. He wanted to play with his cool, but the sun was to happy and because it was so excited to go the car with her. The boy saw the more sun

TEMPERATURE: 0.8
Once upon a time, there was a little girl named Ben. He asked the room and said they is an anything brocks. They had fter a bout the brave and make the balloon. They were so happy and some and pretend.

One day, they like to play with the farmer, she cunted to the will the friends. And said not was buy with backs. She was amazing to all the such and when he saw a big rocke of her toys. She put the door and him an

TEMPERATURE: 1.0
Once upon a time there were twindrell. Don't grand the slipp like and loude with her much fun and still he feets

In [26]:

test_prompts = [
    "Once upon a time",
    "One day, a little girl",
    "A little boy went to",
    "One sunny morning"
]

for prompt in test_prompts:

    print("\n" + "=" * 60)
    print("PROMPT:", prompt)
    print("=" * 60)

    story = generate_with_temperature(
        prompt,
        max_new_tokens=300,
        temperature=0.5
    )

    print(story)



PROMPT: Once upon a time
Once upon a time there was a little girl named Lily. She was very didn't know the car. He thought them was a big friends. She was very sad and started to see the window to the big house. He wanted to the man across and had a countinue and stopped and new to play in the sky.

Once upon a time there was a big box. Sh

PROMPT: One day, a little girl
One day, a little girl named Lily. The was was so always playing in the park.

One day, he was very proped to the sky and saw the farm with a moman and saw a swing his sing. He was so happy and started to her mom and realized to her house and she wanted to play and big house it was a picture of the chouse.

Once upon a t

PROMPT: A little boy went to
A little boy went to play with his and they strong. He was brave and so thrave him to see the little girl named Lily.

Once upon a time, there was a little girl named Timmy. He loved to play with the could not see the window an tried. He was so happy to the park and said,

In [27]:

# Final sample generation

final_prompt = "Once upon a time, there was a little girl"

final_story = generate_with_temperature(
    prompt=final_prompt,
    max_new_tokens=700,
    temperature=0.5
)

print("STORYGPT - FINAL GENERATED STORY")
print("=" * 60)
print(final_story)


STORYGPT - FINAL GENERATED STORY
Once upon a time, there was a little girl named Lily. She never saw a little girl named Lily. She wanted to eat to the car and saw the friends with his friends. She would be goin to the tree and was so strong to eat. She was so happy. He could not be countinued in the park. The bird was so happy and asked her for the forest. He wanted to reach a big back and see the pooke and some to cruck the water.

Once upon a time there was a little girl named Lily. She loved to play with her pretty, she could have and some to go around the book and played in the sunside. He was so three who was brought the bird and the grab home. The boy was story and said, "That is my saw a great of the bug," said. "The cat work and said, "Wow, we are good at
